In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

df1 = pd.read_csv(r"C:\Users\laami\Downloads\order_line\order_info.csv")
df2 = pd.read_csv(r"C:\Users\laami\Downloads\order_line\order_line.csv")
print(df1.head())


   Order ID Customer ID Warehouse ID  Customer Age Customer Gender        Date
0         1     CUST966        WH004            65          Female  2023-03-04
1         2     CUST952        WH003            31          Female  2023-04-04
2         3     CUST987        WH001            25          Female  2023-02-07
3         4     CUST524        WH001            56            Male  2023-03-22
4         5     CUST415        WH002            59            Male  2023-11-11


In [4]:
print(df2.head())

   Order ID   Product ID   SKU ID                  Category  Quantity  \
0      2664  Product_099  SKU_053    Beauty & Personal Care         2   
1      2664  Product_138  SKU_087              Toys & Games         4   
2      2664  Product_190  SKU_831  Groceries & Gourmet Food         4   
3      5028  Product_156  SKU_524    Beauty & Personal Care         3   
4      9143  Product_126  SKU_651    Beauty & Personal Care         2   

   Price per Unit  
0           75.67  
1          636.36  
2           82.44  
3           57.16  
4           35.05  


In [5]:
df = pd.merge(df1, df2, on='Order ID', how='inner')
print(df.head())


   Order ID Customer ID Warehouse ID  Customer Age Customer Gender  \
0         1     CUST966        WH004            65          Female   
1         1     CUST966        WH004            65          Female   
2         1     CUST966        WH004            65          Female   
3         1     CUST966        WH004            65          Female   
4         1     CUST966        WH004            65          Female   

         Date   Product ID   SKU ID           Category  Quantity  \
0  2023-03-04  Product_110  SKU_291  Health & Wellness         4   
1  2023-03-04  Product_142  SKU_005  Health & Wellness         5   
2  2023-03-04  Product_197  SKU_151        Electronics         2   
3  2023-03-04  Product_195  SKU_945  Fashion & Apparel         1   
4  2023-03-04  Product_110  SKU_333  Health & Wellness         5   

   Price per Unit  
0          527.06  
1          847.13  
2          214.60  
3          513.50  
4          436.15  


In [6]:
print(df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 24885 entries, 0 to 24884
Data columns (total 11 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   Order ID         24885 non-null  int64  
 1   Customer ID      24885 non-null  object 
 2   Warehouse ID     24885 non-null  object 
 3   Customer Age     24885 non-null  int64  
 4   Customer Gender  24885 non-null  object 
 5   Date             24885 non-null  object 
 6   Product ID       24885 non-null  object 
 7   SKU ID           24885 non-null  object 
 8   Category         24885 non-null  object 
 9   Quantity         24885 non-null  int64  
 10  Price per Unit   24885 non-null  float64
dtypes: float64(1), int64(3), object(7)
memory usage: 2.1+ MB
None


In [7]:
df['Order value'] = df['Quantity'] * df['Price per Unit']
print(df[['Customer ID', 'Quantity', 'Price per Unit', 'Order value']].head())


  Customer ID  Quantity  Price per Unit  Order value
0     CUST966         4          527.06      2108.24
1     CUST966         5          847.13      4235.65
2     CUST966         2          214.60       429.20
3     CUST966         1          513.50       513.50
4     CUST966         5          436.15      2180.75


In [8]:
avg_order_value = df.groupby('Customer ID', as_index=False)['Order value'].mean()
avg_order_value.rename(columns={'Order value': 'Average Order Value'}, inplace=True)
avg_order_value = avg_order_value.reset_index(drop=True)
print(avg_order_value.head())

  Customer ID  Average Order Value
0     CUST001          1348.963721
1     CUST002          1353.771034
2     CUST003          1187.813929
3     CUST004          1382.176667
4     CUST005          1526.746667


In [9]:
customer_info = df[['Customer ID', 'Customer Age', 'Customer Gender']].drop_duplicates(subset='Customer ID')
customer_info = customer_info.reset_index(drop=True)
print(customer_info.head())

  Customer ID  Customer Age Customer Gender
0     CUST966            65          Female
1     CUST952            31          Female
2     CUST987            25          Female
3     CUST415            59            Male
4     CUST160            36          Female


In [10]:
merged_df = pd.merge(avg_order_value, customer_info, on='Customer ID', how='left')
merged_df = merged_df.reset_index(drop=True)
print(merged_df.head())

  Customer ID  Average Order Value  Customer Age Customer Gender
0     CUST001          1348.963721            24          Female
1     CUST002          1353.771034            35          Female
2     CUST003          1187.813929            26            Male
3     CUST004          1382.176667            25            Male
4     CUST005          1526.746667            29          Female


In [11]:
final_table = merged_df[['Customer Age', 'Customer Gender', 'Average Order Value']]
final_table = final_table.reset_index(drop=True)
print(final_table.head())

   Customer Age Customer Gender  Average Order Value
0            24          Female          1348.963721
1            35          Female          1353.771034
2            26            Male          1187.813929
3            25            Male          1382.176667
4            29          Female          1526.746667


In [12]:
final_table_unique = final_table.drop_duplicates(subset=['Customer Age', 'Customer Gender'])
final_table_unique = final_table_unique.reset_index(drop=True)
print(final_table_unique.head())

   Customer Age Customer Gender  Average Order Value
0            24          Female          1348.963721
1            35          Female          1353.771034
2            26            Male          1187.813929
3            25            Male          1382.176667
4            29          Female          1526.746667


In [13]:
print(final_table_unique.tail())

    Customer Age Customer Gender  Average Order Value
91            58          Female          1291.512667
92            43            Male          1424.547000
93            24            Male           730.531333
94            65          Female          1369.595833
95            44            Male          1346.532500


In [17]:
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import r2_score, mean_squared_error
import pickle
import pandas as pd

df = final_table_unique.copy()
df['Gender'] = LabelEncoder().fit_transform(df['Customer Gender'])  # Female=0, Male=1

X = df[['Customer Age', 'Gender']]
y = df['Average Order Value']

#  Train/Validation/Test Split (60/20/20)
X_train_val, X_test, y_train_val, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
X_train, X_val, y_train, y_val = train_test_split(X_train_val, y_train_val, test_size=0.25, random_state=42)  # 0.25 x 0.8 = 0.2

# parameter grids
param_grids = {
    "Ridge": {"alpha": [0.01, 0.1, 1, 10]},
    "Lasso": {"alpha": [0.01, 0.1, 1, 10]},
    "DecisionTree": {"max_depth": [2, 4, 6, None], "min_samples_split": [2, 5]},
    "RandomForest": {"n_estimators": [10, 50], "max_depth": [4, 6, None], "min_samples_split": [2, 5]}
}

base_models = {
    "LinearRegression": LinearRegression(),
    "Ridge": Ridge(),
    "Lasso": Lasso(),
    "DecisionTree": DecisionTreeRegressor(),
    "RandomForest": RandomForestRegressor()
}

#  Training all the models 
results = {}

for name, model in base_models.items():
    if name == "LinearRegression":
        model.fit(X_train, y_train)
        best_model = model
    else:
        grid = GridSearchCV(model, param_grids[name], cv=5, scoring='r2', n_jobs=-1)
        grid.fit(X_train, y_train)
        best_model = grid.best_estimator_
    
    # Train 
    y_train_pred = best_model.predict(X_train)
    train_r2 = r2_score(y_train, y_train_pred)
    train_mse = mean_squared_error(y_train, y_train_pred)

    # Validation 
    y_val_pred = best_model.predict(X_val)
    val_r2 = r2_score(y_val, y_val_pred)
    val_mse = mean_squared_error(y_val, y_val_pred)

    # Test 
    y_test_pred = best_model.predict(X_test)
    test_r2 = r2_score(y_test, y_test_pred)
    test_mse = mean_squared_error(y_test, y_test_pred)

    results[name] = {
        "model": best_model,
        "train_r2": train_r2,
        "train_mse": train_mse,
        "val_r2": val_r2,
        "val_mse": val_mse,
        "test_r2": test_r2,
        "test_mse": test_mse
    }

results_df = pd.DataFrame(results).T
results_df['r2_rank'] = results_df['val_r2'].rank(ascending=False)
results_df['mse_rank'] = results_df['val_mse'].rank(ascending=True)
results_df['combined_score'] = results_df['r2_rank'] + results_df['mse_rank']

best_model_name = results_df.sort_values('combined_score').index[0]
best_model = results[best_model_name]['model']

print(" Best Model (Validation R² + MSE criteria):", best_model_name)
print(results_df[['train_r2', 'train_mse', 'val_r2', 'val_mse', 'test_r2', 'test_mse', 'combined_score']])

#  Save best model
with open("regression_grid.pkl", "wb") as f:
    pickle.dump(best_model, f)



 Best Model (Validation R² + MSE criteria): Ridge
                  train_r2     train_mse    val_r2       val_mse   test_r2  \
LinearRegression  0.039167  38004.466001 -0.039941  91050.641581 -0.031948   
Ridge             0.032491   38268.52085 -0.018988  89216.207509 -0.017759   
Lasso             0.039065  38008.507166 -0.036987   90792.03966 -0.029536   
DecisionTree      0.388665  24180.539223  -0.09653  96005.243032 -0.467903   
RandomForest      0.529818  18597.424129 -0.087295  95196.702364  0.039386   

                       test_mse  combined_score  
LinearRegression    95370.45592             6.0  
Ridge              94059.172829             2.0  
Lasso              95147.601182             4.0  
DecisionTree      135660.538186            10.0  
RandomForest       88777.927147             8.0  


In [19]:
def predict_new_customer():
    try:
        age = int(input("Enter Customer Age: "))
        gender = input("Enter Customer Gender (Male/Female): ").strip().lower()

        if gender not in ['male', 'female']:
            print("Please enter 'Male' or 'Female'")
            return

        gender_encoded = 0 if gender == "female" else 1
        input_df = pd.DataFrame({"Customer Age": [age], "Gender": [gender_encoded]})

        prediction = best_model.predict(input_df)
        print("Predicted Average Order Value: ₹", round(prediction[0], 2))

    except Exception as e:
        print(" Error:", e)

predict_new_customer()


Enter Customer Age:  56
Enter Customer Gender (Male/Female):  female


Predicted Average Order Value: ₹ 1453.15
